# 使用 Realtime API 进行上下文摘要
## 1. 概述
构建一个端到端的**语音机器人**：它监听你的麦克风，实时用语音回复，并**对长对话进行摘要**，让质量始终不下降。

### 你将学到
1. **实时麦克风流式传输** → OpenAI *Realtime*（语音到语音）端点。
2. 每个轮次的**即时转录和语音播放**。
3. **对话状态容器**，存储**每一条** user/assistant 消息。
4. **自动"上下文裁剪"**——当 token 窗口变得很大时（可配置），较早的轮次会被压缩成摘要。
5. **可扩展的设计**，可改造用于客服机器人、自助终端或多语言助手。


### 前提条件

| 要求 | 详情 |
|-------------|---------|
| **Python ≥ 3.10** | 确保你不会遇到任何问题 |
| **OpenAI API 密钥** | 在 shell 中设置 `OPENAI_API_KEY`，或直接内联粘贴（*不适合生产环境*） |
| 麦克风 + 扬声器 | 如有提示，请授予操作系统权限 |


**需要帮助设置密钥？**  
> 请按照[官方快速入门指南](https://platform.openai.com/docs/quickstart#step-2-set-your-api-key)操作。


*注意：*
> 1. gpt-realtime 支持 32k token 的上下文窗口，但在某些用例中，随着往上下文窗口中塞入更多 token，你可能会注意到性能下降。
> 2. Token 窗口 = 模型当前在会话中保留在记忆中的所有 token（文字和音频 token）。x

### 一行命令安装（在新单元格中运行）

*新的 API 参数：*
> 1. Realtime API 正式版（GA）发布了一个新参数 [`truncation`](https://platform.openai.com/docs/api-reference/realtime-client-events/session/update#realtime-client-events/session/update-session-realtime-session-configuration-truncation)。该参数会自动优化上下文截断，在保留相关信息的同时最大化缓存命中率。

In [ ]:
# Run once to install or upgrade dependencies (comment out if already installed)
# !pip install --upgrade openai websockets sounddevice simpleaudio

In [1]:
# Standard library imports
import os
import sys
import io
import json
import base64
import pathlib
import wave
from dataclasses import dataclass, field
from typing import List, Literal

# Third-party imports
import asyncio
import numpy as np
import sounddevice as sd         # microphone capture
import simpleaudio               # speaker playback
import websockets                # WebSocket client
import openai                    # OpenAI Python SDK >= 1.14.0

In [2]:
# Set your API key safely
openai.api_key = os.getenv("OPENAI_API_KEY", "")
if not openai.api_key:
    raise ValueError("OPENAI_API_KEY not found – please set env var or edit this cell.")

## 2. Token 利用率——文本与语音

大 token 窗口非常宝贵，你多用的每一个 token 都会带来延迟和费用。  
对于**音频**，输入 token 窗口的增长速度比纯文本快得多，因为振幅、时序和其他声学细节都需要被表示出来。

实际上，*同一个*句子，音频形式的 token 数量通常是文本的 **≈ 10 倍**。


* gpt-realtime 最多接受 **32k token**，随着 token 数量增加，模型对指令的遵循可能会出现偏移。
* 每个 user/assistant 轮次都会消耗 token → 窗口**只增不减**。
* **策略**：将较早的轮次摘要为一条 assistant 消息，保留最近几个轮次的原文，然后继续对话。

<img src="../images/text-vs-audio-tokens.png" alt="drawing" width="800"/>

## 3. 辅助函数
以下辅助函数让我们能够运行完整脚本。

### 3.1 对话状态
与基于 HTTP 的 Chat Completions 不同，Realtime API 维护一个开放的**有状态**会话，包含两个关键组件：

| 组件       | 用途 |
|----------------|---------|
| **Session**     | 控制全局设置——模型、语音、模态、VAD 等。 |
| **Conversation** | 存储用户与助手之间逐轮的消息——包括音频和文本。 |

本 notebook 将这些组件封装在一个简单的 `ConversationState` 对象中，以保持逻辑清晰、跟踪历史，并在上下文窗口填满时管理摘要。

In [3]:
@dataclass
class Turn:
    """One utterance in the dialogue (user **or** assistant)."""
    role: Literal["user", "assistant"]
    item_id: str                    # Server‑assigned identifier
    text: str | None = None         # Filled once transcript is ready

@dataclass
class ConversationState:
    """All mutable data the session needs — nothing more, nothing less."""
    history: List[Turn] = field(default_factory=list)         # Ordered log
    waiting: dict[str, asyncio.Future] = field(default_factory=dict)  # Pending transcript fetches
    summary_count: int = 0

    latest_tokens: int = 0          # Window size after last reply
    summarising: bool = False       # Guard so we don’t run two summaries at once

一个用于快速查看转录内容的辅助函数：

In [4]:
def print_history(state) -> None:
    """Pretty-print the running transcript so far."""
    print("—— Conversation so far ———————————————")
    for turn in state.history:
        text_preview = (turn.text or "").strip().replace("\n", " ")
        print(f"[{turn.role:<9}] {text_preview}  ({turn.item_id})")
    print("——————————————————————————————————————————")

### 3.2 · 音频流式传输
我们将把原始 PCM‑16 麦克风数据直接流式传输到 Realtime API。

流水线为：mic ─► async.Queue ─► WebSocket ─► Realtime API

#### 3.2.1 采集麦克风输入
我们先编写一个协程，它会：

* 以 **24 kHz、单声道、PCM‑16**（Realtime 接受的[格式](https://platform.openai.com/docs/api-reference/realtime-sessions/create#realtime-sessions-create-input_audio_format)之一）打开默认麦克风。  
* 将音频流切分成 **≈ 40 ms** 的块。  
* 把每个块放入 `asyncio.Queue`，以便另一个任务（下一节）将其转发给 OpenAI。

In [5]:
async def mic_to_queue(pcm_queue: asyncio.Queue[bytes]) -> None:
    """
    Capture raw PCM‑16 microphone audio and push ~CHUNK_DURATION_MS chunks
    to *pcm_queue* until the surrounding task is cancelled.

    Parameters
    ----------
    pcm_queue : asyncio.Queue[bytes]
        Destination queue for PCM‑16 frames (little‑endian int16).
    """
    blocksize = int(SAMPLE_RATE_HZ * CHUNK_DURATION_MS / 1000)

    def _callback(indata, _frames, _time, status):
        if status:                               # XRuns, device changes, etc.
            print("⚠️", status, file=sys.stderr)
        try:
            pcm_queue.put_nowait(bytes(indata))  # 1‑shot enqueue
        except asyncio.QueueFull:
            # Drop frame if upstream (WebSocket) can’t keep up.
            pass

    # RawInputStream is synchronous; wrap in context manager to auto‑close.
    with sd.RawInputStream(
        samplerate=SAMPLE_RATE_HZ,
        blocksize=blocksize,
        dtype="int16",
        channels=1,
        callback=_callback,
    ):
        try:
            # Keep coroutine alive until cancelled by caller.
            await asyncio.Event().wait()
        finally:
            print("⏹️  Mic stream closed.")

#### 3.2.2 将音频块发送到 API

我们的麦克风任务现在正在向 `asyncio.Queue` 填充原始 PCM‑16 块。  
下一步：从队列中取出音频块，对其进行 **base‑64 编码**（协议要求 JSON 安全的文本），然后将每个块作为 `input_audio_buffer.append` 事件发送到 Realtime WebSocket。

In [6]:
# Helper function to encode audio chunks in base64
b64 = lambda blob: base64.b64encode(blob).decode()

async def queue_to_websocket(pcm_queue: asyncio.Queue[bytes], ws):
    """Read audio chunks from queue and send as JSON events."""
    try:
        while (chunk := await pcm_queue.get()) is not None:
            await ws.send(json.dumps({
                "type": "input_audio_buffer.append",
                "audio": b64(chunk),
            }))
    except websockets.ConnectionClosed:
        print("WebSocket closed – stopping uploader")

#### 3.2.3 处理传入事件 
音频到达服务器后，Realtime API 会通过**同一个** WebSocket 推回一连串 JSON 事件。  
理解这些事件对于以下几点至关重要：

* 打印实时转录  
* 向用户播放增量音频  
* 维护准确的 [`Conversation State`](https://platform.openai.com/docs/api-reference/realtime-server-events/conversation/created)，以便后续上下文裁剪正常工作  


| 事件类型 | 何时到达 | 为何重要 | 典型处理逻辑 |
|------------|-----------------|---------------|-----------------------|
| **`session.created`** | WebSocket 握手完成后立即到达 | 确认会话已打开，并提供 `session.id`。 | 记录该 ID 以便追踪，并验证连接。 |
| **`session.updated`** | 发送 `session.update` 调用之后 | 确认服务器已应用新的会话设置。 | 检查回显的设置，并更新本地缓存。 |
| **`conversation.item.created`**（user） | 用户停止说话后几毫秒（客户端 VAD 触发） | 预留一个时间线位置；转录可能仍为 **`null`**。 | 在 `state.history` 中插入一个标记为"等待转录"的*占位*用户轮次。 |
| **`conversation.item.retrieved`** | 约 100 – 300 ms 后，音频转录完成时 | 提供最终的用户转录（含时间信息）。 | 用转录替换占位内容，如有需要可将其打印出来。 |
| **`response.audio.delta`** | 助手说话期间每 20 – 60 ms 一次 | 流式传输 PCM‑16 音频块（以及可选的增量文本）。 | 缓冲每个音频块并播放；可选择在控制台显示部分文本。 |
| **`response.done`** | 助手的最后一个 token 之后 | 表示音频和文本都已完成；包含用量统计。 | 完成助手轮次，更新 `state.latest_tokens`，并记录用量。 |
| **`conversation.item.deleted`** | 每当你使用 `conversation.item.delete` 进行裁剪时 | 确认某个轮次已被删除，释放服务器上的 token。 | 在本地同步执行删除，使你的上下文窗口与服务器保持一致。 |



### 3.3 检测何时进行摘要
Realtime 模型拥有 **32k token 的大窗口**，但随着你往模型中塞入更多上下文，质量可能在远未达到该上限时就出现偏移。

我们的目标：一旦当前窗口接近安全阈值（本 notebook 中默认为 **2 000 个 token**），就**自动进行摘要**，然后在本地*和*服务端同时裁剪掉被取代的轮次。

我们监控 `response.done` 中返回的 latest_tokens。当它超过 SUMMARY_TRIGGER 且轮次数多于 KEEP_LAST_TURNS 时，就启动一个后台摘要协程。

我们把除最后 2 个轮次之外的所有内容压缩成一段法语段落，然后：

1. 将该段落作为一条新的 assistant 消息插入到对话的最前面。

2. 删除用于生成摘要的那些消息条目。

之后，我们会询问语音智能体摘要使用的是什么语言，以测试摘要是否已成功插入 Realtime API 的对话上下文。

In [7]:
async def run_summary_llm(text: str) -> str:
    """Call a lightweight model to summarise `text`."""
    resp = await asyncio.to_thread(lambda: openai.chat.completions.create(
        model=SUMMARY_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "Summarise in French the following conversation "
                            "in one concise paragraph so it can be used as "
                            "context for future dialogue."},
            {"role": "user", "content": text},
        ],
    ))
    return resp.choices[0].message.content.strip()

重要的实现细节：
- 摘要是作为 SYSTEM 消息而不是 ASSISTANT 消息追加的。测试表明，在长时间对话中，使用 ASSISTANT 消息作为摘要可能导致模型错误地从音频回复切换为文本回复。通过使用 SYSTEM 消息作为摘要（其中还可以包含额外的自定义指令），我们清楚地向模型表明这些是用于设定上下文的指令，从而防止它错误地沿用正在进行的用户-助手交互的模态。

In [8]:
async def summarise_and_prune(ws, state):
    """Summarise old turns, delete them server‑side, and prepend a single summary
    turn locally + remotely."""
    state.summarising = True
    print(
        f"⚠️  Token window ≈{state.latest_tokens} ≥ {SUMMARY_TRIGGER}. Summarising…",
    )
    old_turns, recent_turns = state.history[:-KEEP_LAST_TURNS], state.history[-KEEP_LAST_TURNS:]
    convo_text = "\n".join(f"{t.role}: {t.text}" for t in old_turns if t.text)
    
    if not convo_text:
        print("Nothing to summarise (transcripts still pending).")
        state.summarising = False

    summary_text = await run_summary_llm(convo_text) if convo_text else ""
    state.summary_count += 1
    summary_id = f"sum_{state.summary_count:03d}"
    state.history[:] = [Turn("assistant", summary_id, summary_text)] + recent_turns
    
    print_history(state)    

    # Create summary on server
    await ws.send(json.dumps({
        "type": "conversation.item.create",
        "previous_item_id": "root",
        "item": {
            "id": summary_id,
            "type": "message",
            "role": "system",
            "content": [{"type": "input_text", "text": summary_text}],
        },
    }))

    # Delete old items
    for turn in old_turns:
        await ws.send(json.dumps({
            "type": "conversation.item.delete",
            "item_id": turn.item_id,
        }))

    print(f"✅ Summary inserted ({summary_id})")
    
    state.summarising = False

下面的函数让我们可以随时间轮询转录结果。当用户的音频没有被立即转录时，这很有用，我们可以稍后再获取最终结果。

In [9]:
async def fetch_full_item(
    ws, item_id: str, state: ConversationState, attempts: int = 1
):
    """
    Ask the server for a full conversation item; retry up to 5× if the
    transcript field is still null.  Resolve the waiting future when done.
    """
    # If there is already a pending fetch, just await it
    if item_id in state.waiting:
        return await state.waiting[item_id]

    fut = asyncio.get_running_loop().create_future()
    state.waiting[item_id] = fut

    await ws.send(json.dumps({
        "type": "conversation.item.retrieve",
        "item_id": item_id,
    }))
    item = await fut

    # If transcript still missing retry (max 5×)
    if attempts < 5 and not item.get("content", [{}])[0].get("transcript"):
        await asyncio.sleep(0.4 * attempts)
        return await fetch_full_item(ws, item_id, state, attempts + 1)

    # Done – remove the marker
    state.waiting.pop(item_id, None)
    return item


## 4. 端到端工作流演示

运行下面两个单元格以启动交互式会话。中断单元格即可停止录音。

> **注意：**  
> 本 notebook 使用 `SUMMARY_TRIGGER = 2000` 和 `KEEP_LAST_TURNS = 2`，以便快速演示摘要功能。  
> 在生产环境中，你应该根据应用的需求调整这些值。  
> - 典型的 `SUMMARY_TRIGGER` 介于 **20,000–32,000 个 token** 之间，具体取决于在你的用例中性能随上下文增大而下降的情况。

In [10]:
# Audio/config knobs
SAMPLE_RATE_HZ    = 24_000   # Required by pcm16
CHUNK_DURATION_MS = 40       # chunk size for audio capture
BYTES_PER_SAMPLE  = 2        # pcm16 = 2 bytes/sample
SUMMARY_TRIGGER   = 2_000    # Summarise when context ≥ this
KEEP_LAST_TURNS   = 2       # Keep these turns verbatim
SUMMARY_MODEL     = "gpt-4o-mini"  # Cheaper, fast summariser

In [11]:
# --------------------------------------------------------------------------- #
# Realtime session                                                          #
# --------------------------------------------------------------------------- #
async def realtime_session(model="gpt-realtime", voice="shimmer", enable_playback=True):
    """
    Main coroutine: connects to the Realtime endpoint, spawns helper tasks,
    and processes incoming events in a big async‑for loop.
    """
    state = ConversationState()  # Reset state for each run

    pcm_queue: asyncio.Queue[bytes] = asyncio.Queue()
    assistant_audio: List[bytes] = []

    # ----------------------------------------------------------------------- #
    # Open the WebSocket connection to the Realtime API                       #
    # ----------------------------------------------------------------------- #
    url = f"wss://api.openai.com/v1/realtime?model={model}"
    headers = {"Authorization": f"Bearer {openai.api_key}"}

    async with websockets.connect(url, extra_headers=headers, max_size=1 << 24) as ws:
        # ------------------------------------------------------------------- #
        # Wait until server sends session.created                             #
        # ------------------------------------------------------------------- #
        while json.loads(await ws.recv())["type"] != "session.created":
            pass
        print("session.created ✅")

        # ------------------------------------------------------------------- #
        # Configure session: voice, modalities, audio formats, transcription  #
        # ------------------------------------------------------------------- #
        await ws.send(json.dumps({
            "type": "session.update",
            "session": {
                "type": "realtime",
                model: "gpt-realtime",
                "voice": voice,
                "modalities": ["audio", "text"],
                "input_audio_format": "pcm16",
                "output_audio_format": "pcm16",
                "input_audio_transcription": {"model": "gpt-4o-transcribe"},
            },
        }))

        # ------------------------------------------------------------------- #
        # Launch background tasks: mic capture → queue → websocket            #
        # ------------------------------------------------------------------- #
        mic_task = asyncio.create_task(mic_to_queue(pcm_queue))
        upl_task = asyncio.create_task(queue_to_websocket(pcm_queue, ws))

        print("🎙️ Speak now (Ctrl‑C to quit)…")

        try:
            # ------------------------------------------------------------------- #
            # Main event loop: process incoming events from the websocket         #
            # ------------------------------------------------------------------- #
            async for event_raw in ws:
                event = json.loads(event_raw)
                etype = event["type"]

                # --------------------------------------------------------------- #
                # User just spoke ⇢ conversation.item.created (role = user)        #
                # --------------------------------------------------------------- #
                if etype == "conversation.item.created" and event["item"]["role"] == "user":
                    item = event["item"]
                    text = None
                    if item["content"]:
                        text = item["content"][0].get("transcript")
                    
                    state.history.append(Turn("user", event["item"]["id"], text))
                    
                    # If transcript not yet available, fetch it later
                    if text is None:
                        asyncio.create_task(fetch_full_item(ws, item["id"], state))

                # --------------------------------------------------------------- #
                # Transcript fetched ⇢ conversation.item.retrieved                 #
                # --------------------------------------------------------------- #
                elif etype == "conversation.item.retrieved":
                    content = event["item"]["content"][0]
                    # Fill missing transcript in history
                    for t in state.history:
                        if t.item_id == event["item"]["id"]:
                            t.text = content.get("transcript")
                            break

                # --------------------------------------------------------------- #
                # Assistant audio arrives in deltas                               #
                # --------------------------------------------------------------- #
                elif etype == "response.audio.delta":
                    assistant_audio.append(base64.b64decode(event["delta"]))

                # --------------------------------------------------------------- #
                # Assistant reply finished ⇢ response.done                        #
                # --------------------------------------------------------------- #
                elif etype == "response.done":
                    for item in event["response"]["output"]:
                        if item["role"] == "assistant":
                            txt = item["content"][0]["transcript"]
                            state.history.append(Turn("assistant", item["id"], txt))
                            # print(f"\n🤖 {txt}\n")
                    state.latest_tokens = event["response"]["usage"]["total_tokens"]
                    print(f"—— response.done  (window ≈{state.latest_tokens} tokens) ——")
                    print_history(state)
                    
                    # Fetch any still‑missing user transcripts
                    for turn in state.history:
                        if (turn.role == "user"
                            and turn.text is None
                            and turn.item_id not in state.waiting):
                            asyncio.create_task(
                                fetch_full_item(ws, turn.item_id, state)
                            )

                    # Playback collected audio once reply completes
                    if enable_playback and assistant_audio:
                        simpleaudio.play_buffer(b"".join(assistant_audio), 1, BYTES_PER_SAMPLE, SAMPLE_RATE_HZ)
                        assistant_audio.clear()

                    # Summarise if context too large – fire in background so we don't block dialogue
                    if state.latest_tokens >= SUMMARY_TRIGGER and len(state.history) > KEEP_LAST_TURNS and not state.summarising:
                        asyncio.create_task(summarise_and_prune(ws, state))

        except KeyboardInterrupt:
            print("\nStopping…")
        finally:
            mic_task.cancel()
            await pcm_queue.put(None)
            await upl_task

In [ ]:
# Run the realtime session (this cell blocks until you stop it)
await realtime_session()

```raw
session.created ✅
🎙️ Speak now (Ctrl‑C to quit)…
—— response.done  (window ≈979 tokens) ——
—— Conversation so far ———————————————
[user     ] Can you tell me a quick story?  (item_BTuMOcpUqp8qknKhLzlkA)
[assistant] Once upon a time, in a cozy little village, there was a cat named Whiskers who was always getting into trouble. One sunny day, Whiskers found a mysterious glowing stone in the garden. Curious, he pawed at it, and poof! The stone granted him the ability to talk to birds. Whiskers and his new bird friends had grand adventures, solving mysteries and exploring the village. And from that day on, Whiskers was known as the most adventurous cat in the village. The end.  (item_BTuMPRWxqpv0ph6QM46DK)
——————————————————————————————————————————
—— response.done  (window ≈2755 tokens) ——
—— Conversation so far ———————————————
[user     ] Can you tell me a quick story?  (item_BTuMOcpUqp8qknKhLzlkA)
[assistant] Once upon a time, in a cozy little village, there was a cat named Whiskers who was always getting into trouble. One sunny day, Whiskers found a mysterious glowing stone in the garden. Curious, he pawed at it, and poof! The stone granted him the ability to talk to birds. Whiskers and his new bird friends had grand adventures, solving mysteries and exploring the village. And from that day on, Whiskers was known as the most adventurous cat in the village. The end.  (item_BTuMPRWxqpv0ph6QM46DK)
[user     ] Can you tell me three extremely funny stories?  (item_BTuNN64LdULM21OyC4vzN)
[assistant] Sure, let's dive into some giggle-worthy tales:  **Story One:** There was a forgetful baker named Benny who baked a hundred cakes for a big wedding. But on the big day, he forgot where he put them! The entire town joined in to find the missing cakes, only to discover Benny had stored them in his neighbor's garage, thinking it was his pantry. The wedding turned into a town-wide cake feast!  **Story Two:** A mischievous dog named Sparky loved to play pranks. One day, he swapped his owner's phone with a squeaky toy, causing a hilarious mix-up of barks, squeaks, and confused calls. Sparky's owner ended up having a full conversation with the mailman, all in squeaks!  **Story Three:** In a small town, a parrot named Polly became a local celebrity for reciting tongue twisters. One day, Polly challenged the mayor to a tongue twister duel. The mayor, tongue-tied and laughing, declared Polly the official town jester. Polly squawked with pride, and the town rang with laughter for days.  (item_BTuNNpNxki5ynSQ5c3Xsa)
——————————————————————————————————————————
⚠️  Token window ≈2755 ≥ 2000. Summarising…
—— Conversation so far ———————————————
[assistant] L'utilisateur a demandé une histoire rapide, et l'assistant a raconté celle d'un chat nommé Whiskers qui, après avoir trouvé une pierre mystérieuse dans son jardin, a obtenu le pouvoir de parler aux oiseaux. Avec ses nouveaux amis oiseaux, Whiskers a vécu de grandes aventures, résolvant des mystères et explorant le village, devenant ainsi le chat le plus aventurier du village.  (sum_001)
[user     ] Can you tell me three extremely funny stories?  (item_BTuNN64LdULM21OyC4vzN)
[assistant] Sure, let's dive into some giggle-worthy tales:  **Story One:** There was a forgetful baker named Benny who baked a hundred cakes for a big wedding. But on the big day, he forgot where he put them! The entire town joined in to find the missing cakes, only to discover Benny had stored them in his neighbor's garage, thinking it was his pantry. The wedding turned into a town-wide cake feast!  **Story Two:** A mischievous dog named Sparky loved to play pranks. One day, he swapped his owner's phone with a squeaky toy, causing a hilarious mix-up of barks, squeaks, and confused calls. Sparky's owner ended up having a full conversation with the mailman, all in squeaks!  **Story Three:** In a small town, a parrot named Polly became a local celebrity for reciting tongue twisters. One day, Polly challenged the mayor to a tongue twister duel. The mayor, tongue-tied and laughing, declared Polly the official town jester. Polly squawked with pride, and the town rang with laughter for days.  (item_BTuNNpNxki5ynSQ5c3Xsa)
——————————————————————————————————————————
✅ Summary inserted (sum_001)
—— response.done  (window ≈2147 tokens) ——
—— Conversation so far ———————————————
[assistant] L'utilisateur a demandé une histoire rapide, et l'assistant a raconté celle d'un chat nommé Whiskers qui, après avoir trouvé une pierre mystérieuse dans son jardin, a obtenu le pouvoir de parler aux oiseaux. Avec ses nouveaux amis oiseaux, Whiskers a vécu de grandes aventures, résolvant des mystères et explorant le village, devenant ainsi le chat le plus aventurier du village.  (sum_001)
[user     ] Can you tell me three extremely funny stories?  (item_BTuNN64LdULM21OyC4vzN)
[assistant] Sure, let's dive into some giggle-worthy tales:  **Story One:** There was a forgetful baker named Benny who baked a hundred cakes for a big wedding. But on the big day, he forgot where he put them! The entire town joined in to find the missing cakes, only to discover Benny had stored them in his neighbor's garage, thinking it was his pantry. The wedding turned into a town-wide cake feast!  **Story Two:** A mischievous dog named Sparky loved to play pranks. One day, he swapped his owner's phone with a squeaky toy, causing a hilarious mix-up of barks, squeaks, and confused calls. Sparky's owner ended up having a full conversation with the mailman, all in squeaks!  **Story Three:** In a small town, a parrot named Polly became a local celebrity for reciting tongue twisters. One day, Polly challenged the mayor to a tongue twister duel. The mayor, tongue-tied and laughing, declared Polly the official town jester. Polly squawked with pride, and the town rang with laughter for days.  (item_BTuNNpNxki5ynSQ5c3Xsa)
[user     ]   (item_BTuPLaCv8ATdIwAQ2rLgO)
[assistant] Sure! The first summary I provided between us was in French.  (item_BTuPLa7BaSQToGCVOmfBK)
```

---
我们与语音 AI 进行了一次对话。几个轮次之后，总 token 数达到了 SUMMARY_MAX，触发了对话摘要步骤，为较早的消息生成了摘要。

由于总共有 N = 4 条消息，我们对前 N - 2 = 2 条消息进行了摘要：
```txt
—— Conversation so far ———————————————
[user     ] Can you tell me a quick story?  (item_BTuMOcpUqp8qknKhLzlkA)
[assistant] Once upon a time, in a cozy little village, there was a cat named Whiskers who was always getting into trouble. One sunny day, Whiskers found a mysterious glowing stone in the garden. Curious, he pawed at it, and poof! The stone granted him the ability to talk to birds. Whiskers and his new bird friends had grand adventures, solving mysteries and exploring the village. And from that day on, Whiskers was known as the most adventurous cat in the village. The end.  (item_BTuMPRWxqpv0ph6QM46DK)
```

然后，我们用法语生成摘要，并使用 root: true 标志将其插入对话历史，确保摘要作为对话中的第一条消息出现。之后，我们使用 `"type": "conversation.item.delete"` 删除了被摘要的原始条目。

为了验证摘要是否插入成功，我们询问语音 AI 摘要使用的是什么语言。它正确地回答：

```txt
[assistant] Sure! The first summary I provided between us was in French.  (item_BTuPLa7BaSQToGCVOmfBK)
```

## 5 · 实际应用

上下文摘要对于**长时间运行的语音体验**很有用。  
以下是一些用例设想：

| 用例 | 附加价值 | 为何有用 |
|----------|-------------|------------|
| **客服语音机器人** | 7×24 小时的自然语音电话菜单；自动生成工单摘要 | 对冗长的客户来电进行摘要，便于高效交接和记录，减轻客服人员的工作量并提升回复质量。 |
| **语言辅导** | 带纠错反馈的实时对话练习 | 帮助跟踪学习者的进度并突出反复出现的错误，从而提供个性化反馈，更有效地习得语言。 |
| **AI 心理咨询师 / 教练** | 安全、随时可用、能记住历次会话的倾听者 | 通过回忆关键话题和情绪基调，在多次会话之间保持连续性，带来更具同理心、更有效的体验。 |
| **会议助手** | 实时转录 + 在 Slack 中发送简明的待办事项回顾 | 将冗长的会议提炼为可执行的摘要，节省团队成员的时间，并确保不会遗漏要点。 |


## 6 · 后续步骤与延伸阅读
试用本 notebook，并尝试将上下文摘要集成到你的应用中。

你可以尝试以下几件事：
| 尝试这个… | 你将了解到 |
|-----------|------------------|
| **对摘要进行 A/B 测试**<br/>分别在*开启*和*关闭*摘要的情况下运行你的评估套件。 | 裁剪是否真的能提升你所在领域的质量——以及它对延迟和成本的影响。 |
| **更换摘要风格**<br/>将系统提示改为要点列表、JSON、英语或法语等。 | 下游助手最容易吸收哪种格式；语言选择如何影响后续回答。 |
| **调整阈值**<br/>尝试不同的 `SUMMARY_TRIGGER_TOKENS`（2k → 8k）。 | 在模型偏移与摘要开销之间找到最佳平衡点。 |
| **成本追踪**<br/>在摘要前后记录 `usage.total_tokens`。 | 具体的投资回报：每小时对话节省的 token 数。 |


### 资源：
- [OpenAI Realtime 指南](https://platform.openai.com/docs/guides/realtime)
- [OpenAI Realtime 对话](https://platform.openai.com/docs/guides/realtime-conversations)
- [OpenAI Realtime API 参考](https://platform.openai.com/docs/api-reference/realtime)
- [Voice AI and Voice Agents](https://voiceaiandvoiceagents.com/)